# QAOA for Max-Cut: landscape, warm starts, and hardware

| | |
|---|---|
| **Level** | Advanced |
| **Time** | 60 to 90 minutes |
| **Prerequisites** | Max-Cut, QAOA, classical optimization. The Goemans-Williamson section is optional. |
| **Default devices** | IQM Garnet |
| **Hardware jobs** | 1 |
| **Approximate cost** | about 320 credits on Garnet at 2000 shots |
| **Hardware notes** | In our tests Garnet gave an average cut of 10.3, above the random baseline of 9.0. On Rigetti Cepheus the result was at or below the random value in two runs. |

Shot counts for the hardware runs are set at the end of the **Setup** cell. The devices are set in the first hardware cell. Credits are charged only when a hardware cell runs.

*QUEST intermediate and advanced series: Quantum Machine Learning*

The Quantum Approximate Optimization Algorithm (QAOA) targets combinatorial optimization. Its shallow version fits on current hardware, and it has provable structural properties, which makes it worth studying while its practical performance is still debated.

This notebook applies QAOA to Max-Cut: split the vertices of a graph into two sets so that as many edges as possible cross between them. Max-Cut is NP-hard in general and has a well-known classical method, Goemans-Williamson (1995), with a proven approximation ratio of 0.878.

We build QAOA with one layer ($p=1$), plot its parameter landscape, warm-start it from a classical solution, run it on real hardware, and compare with the classical methods. The question throughout is whether QAOA at this depth does anything useful.

**Learning objectives**

1. Write Max-Cut as a Hamiltonian minimization problem.
2. Describe QAOA as alternating cost and mixer layers.
3. Plot the QAOA cost landscape and see why it is hard to optimize.
4. Warm-start QAOA from a classical solution and see how that changes convergence.
5. Run QAOA on real hardware and measure the approximation ratio achieved.
6. (Optional) Compare with Goemans-Williamson.

**Background needed:** basic graph theory (vertices, edges, adjacency). Variational algorithms help, but the pattern is explained.


## Max-Cut and its Hamiltonian formulation

Given an undirected graph $G = (V, E)$ with $|V| = n$ vertices, a **cut** partitions $V$ into two disjoint subsets $S$ and $V \setminus S$. The **size of the cut** is the number of edges with one endpoint in each subset:

$$\text{Cut}(S) = \left| \{(i, j) \in E : i \in S, j \notin S\} \right|$$

**Max-Cut** is the optimization problem of finding the partition that maximizes the cut size.

To encode this on a quantum computer, assign a binary variable $z_i \in \{-1, +1\}$ to each vertex. Vertex $i$ is in $S$ if $z_i = +1$, and in $V \setminus S$ if $z_i = -1$. Then the cut size is:

$$\text{Cut}(z) = \sum_{(i, j) \in E} \frac{1 - z_i z_j}{2}$$

Each edge contributes 1 to the cut if $z_i z_j = -1$ (endpoints on different sides), or 0 if $z_i z_j = +1$ (same side).

To turn this into a Hamiltonian, replace $z_i$ with the Pauli $\hat{Z}_i$ operator on qubit $i$. Then maximizing the cut is equivalent to minimizing the **Max-Cut Hamiltonian**:

$$\hat{H}_{C} = \sum_{(i, j) \in E} \frac{\hat{Z}_i \hat{Z}_j - I}{2}$$

The ground state of $\hat{H}_C$ is a computational basis state whose bit pattern encodes the maximum cut.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx

from scipy.optimize import minimize

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

print("Setup complete.")

# Shot counts for the hardware runs. More shots reduce statistical error but cost
# more on most devices; the README lists prices.
SHOTS = 2000
QUEST_JOB_TAGS = {"quest": "ml-qaoa"}   # labels this notebook's hardware jobs for QUEST usage statistics


## A small random graph

We use an 8-vertex random graph with edge probability 0.5. It's non-trivial, but small enough that we can enumerate all $2^8 = 256$ possible cuts for reference. Larger graphs would require more qubits and more classical benchmark work.


In [ ]:
N_VERTICES = 8
G = nx.erdos_renyi_graph(n=N_VERTICES, p=0.5, seed=7)
edges = list(G.edges())

print(f"Graph: {N_VERTICES} vertices, {len(edges)} edges")
print(f"Edges: {edges}")

fig, ax = plt.subplots(figsize=(6, 5))
pos = nx.spring_layout(G, seed=7)
nx.draw(G, pos, ax=ax, with_labels=True, node_color='#f5e0ee', node_size=800,
        edge_color='#5a4a55', font_size=12, font_weight='bold')
ax.set_title(f'Random graph: {N_VERTICES} vertices, {len(edges)} edges')
plt.tight_layout()
plt.show()

## Exact Max-Cut by enumeration

At $n=8$, we can enumerate all $2^n = 256$ possible bit strings and find the one with the largest cut size. This is our exact reference.

*Aside on scaling:* enumeration is $O(2^n \cdot |E|)$. At $n=20$ it's about $10^6 \cdot |E|$, still tractable. At $n=40$ it's $10^{12} \cdot |E|$, beyond a laptop. This is roughly where classical heuristics take over from exact methods.


In [ ]:
def cut_size(bitstring, edges):
    """Return the cut size for a given bitstring assignment."""
    return sum(1 for (i, j) in edges if bitstring[i] != bitstring[j])


def exact_maxcut(N, edges):
    """Enumerate all 2^N bitstrings, return max cut and optimal bitstring."""
    best_cut = 0
    best_bs = None
    for x in range(2 ** N):
        bs = format(x, f'0{N}b')
        c = cut_size(bs, edges)
        if c > best_cut:
            best_cut = c
            best_bs = bs
    return best_cut, best_bs


max_cut_exact, opt_bitstring = exact_maxcut(N_VERTICES, edges)
print(f"Exact Max-Cut value: {max_cut_exact}")
print(f"Optimal bitstring:   {opt_bitstring}")
print(f"(Ratio to |E|: {max_cut_exact / len(edges):.3f})")

## Classical baseline 1: random cut

The simplest possible algorithm: assign each vertex to a random side, uniformly at random. Expected cut value = $|E| / 2$. Approximation ratio: 0.5.

This is the floor. Any algorithm worth mentioning must beat it.


In [ ]:
def random_cut_expected(edges):
    """Expected value of a uniformly random cut is |E| / 2."""
    return len(edges) / 2


exp_random = random_cut_expected(edges)
print(f"Random cut expected value: {exp_random}")
print(f"Random cut approximation ratio: {exp_random / max_cut_exact:.3f}")

## Optional, advanced: Classical baseline 2: Goemans-Williamson
The Goemans-Williamson algorithm (1995) uses semidefinite programming to construct a cut with expected approximation ratio at least $0.87856$. This is the best known worst-case guarantee for a polynomial-time classical algorithm on Max-Cut, and it's provably tight under the Unique Games Conjecture.

We implement a simplified version: solve the SDP relaxation, then round using a random hyperplane. For pedagogy, we do the SDP with `cvxpy` (a real convex optimization library). If cvxpy isn't installed, comment this section out and use a greedy heuristic instead.


In [ ]:
try:
    import cvxpy as cp

    def goemans_williamson(N, edges, n_rounds=100):
        """
        Solve the Max-Cut SDP relaxation and round using random hyperplanes.
        Return the best cut found across n_rounds of rounding.
        """
        # Build the Laplacian-based objective matrix
        W = np.zeros((N, N))
        for (i, j) in edges:
            W[i, j] = 1
            W[j, i] = 1

        # SDP: maximize sum_{ij} W_ij * (1 - X_ij) / 2, subject to X psd, X_ii = 1
        X = cp.Variable((N, N), symmetric=True)
        constraints = [X >> 0] + [X[i, i] == 1 for i in range(N)]
        objective = cp.Maximize(cp.sum(cp.multiply(W, (1 - X))) / 2)
        prob = cp.Problem(objective, constraints)
        prob.solve()

        # Factor X = V V^T
        X_val = X.value
        # Nudge to ensure positive semi-definite for factorization
        X_val = (X_val + X_val.T) / 2
        eigenvalues, eigenvectors = np.linalg.eigh(X_val)
        eigenvalues = np.maximum(eigenvalues, 0)
        V = eigenvectors * np.sqrt(eigenvalues)

        # Random hyperplane rounding
        best_cut = 0
        best_bs = None
        rng = np.random.default_rng(42)
        for _ in range(n_rounds):
            r = rng.normal(size=N)
            bs = ''.join(['0' if V[i] @ r > 0 else '1' for i in range(N)])
            c = cut_size(bs, edges)
            if c > best_cut:
                best_cut = c
                best_bs = bs

        return best_cut, best_bs

    gw_cut, gw_bitstring = goemans_williamson(N_VERTICES, edges)
    print(f"Goemans-Williamson best cut (from 100 rounding trials): {gw_cut}")
    print(f"GW bitstring:  {gw_bitstring}")
    print(f"GW approximation ratio: {gw_cut / max_cut_exact:.3f}")
    HAVE_GW = True

except ImportError:
    print("cvxpy not available, skipping Goemans-Williamson. Use pip install cvxpy to enable.")
    gw_cut = None
    gw_bitstring = None
    HAVE_GW = False

## QAOA in one section

QAOA prepares a parameterized quantum state:

$$|\psi(\vec{\gamma}, \vec{\beta})\rangle = \prod_{k=1}^{p} e^{-i \beta_k \hat{H}_M} e^{-i \gamma_k \hat{H}_C} |+\rangle^{\otimes n}$$

where:

- $|+\rangle^{\otimes n}$ is the equal superposition over all bit strings (initialized with Hadamards).
- $\hat{H}_C$ is the cost Hamiltonian (our Max-Cut Hamiltonian).
- $\hat{H}_M = \sum_i \hat{X}_i$ is the **mixer** (transverse field), which shuffles bit strings around.
- $\vec{\gamma}, \vec{\beta}$ are $2p$ trainable parameters.

At $p = 1$, only 2 parameters $(\gamma, \beta)$. This is small enough that we can *visualize* the entire cost landscape as a 2D heatmap, which we'll do in a moment. This visualization is one of the most useful teaching artifacts in QAOA.

The expected cost is $\langle \psi(\gamma, \beta) | \hat{H}_C | \psi(\gamma, \beta) \rangle$. We want to *minimize* it (equivalently, maximize the negative), and then measure the state to get bit strings that hopefully correspond to good cuts.


In [ ]:
def build_cost_hamiltonian(N, edges):
    """Return the Max-Cut cost Hamiltonian: sum of (Z_i Z_j - I) / 2 over edges."""
    pauli_list = []
    for (i, j) in edges:
        label = ['I'] * N
        label[i] = 'Z'
        label[j] = 'Z'
        pauli_list.append((''.join(reversed(label)), 0.5))
        # -I/2 term (constant offset, applied once per edge)
        pauli_list.append(('I' * N, -0.5))
    return SparsePauliOp.from_list(pauli_list).simplify()


def qaoa_circuit(gamma, beta, N, edges):
    """Build p=1 QAOA circuit."""
    qc = QuantumCircuit(N)
    # Initial superposition
    for q in range(N):
        qc.h(q)
    # Cost layer: e^(-i gamma H_C) implemented per-edge as R_ZZ(2*gamma)
    for (i, j) in edges:
        qc.rzz(2 * gamma, i, j)
    # Mixer layer: e^(-i beta H_M) implemented per-qubit as R_X(2*beta)
    for q in range(N):
        qc.rx(2 * beta, q)
    return qc


def qaoa_expected_cost(gamma, beta, N, edges, H_cost):
    """Compute <psi(gamma, beta)|H_C|psi(gamma, beta)> exactly."""
    qc = qaoa_circuit(gamma, beta, N, edges)
    sv = Statevector.from_instruction(qc)
    return np.real(sv.expectation_value(H_cost))


H_cost = build_cost_hamiltonian(N_VERTICES, edges)
print(f"Cost Hamiltonian: {len(H_cost)} Pauli terms after simplification")

# Sanity check at random parameters
test_gamma, test_beta = 0.5, 0.7
expected = qaoa_expected_cost(test_gamma, test_beta, N_VERTICES, edges, H_cost)
print(f"<H_C> at (gamma={test_gamma}, beta={test_beta}): {expected:.4f}")

## Visualize the QAOA landscape

At $p = 1$, we have only 2 parameters, so we can plot the entire cost landscape as a 2D heatmap. Most QAOA tutorials skip this plot, which is a shame. Seeing the landscape once teaches you why QAOA optimization is hard.

Parameters live in bounded ranges: $\gamma \in [0, \pi]$ and $\beta \in [0, \pi/2]$ suffice by periodicity.


In [ ]:
gammas = np.linspace(0, np.pi, 40)
betas = np.linspace(0, np.pi / 2, 30)

landscape = np.zeros((len(gammas), len(betas)))
for i, g in enumerate(gammas):
    for j, b in enumerate(betas):
        landscape[i, j] = qaoa_expected_cost(g, b, N_VERTICES, edges, H_cost)

# Find the minimum
min_idx = np.unravel_index(np.argmin(landscape), landscape.shape)
gamma_opt_lin = gammas[min_idx[0]]
beta_opt_lin = betas[min_idx[1]]
best_cost = landscape[min_idx]

# The cost we minimize is <H_C>; the cut value is |E| + <H_C> (since H_C = sum (ZZ - I) / 2, and -1 * H_C is (sum I - ZZ)/2 = # of cut edges)
# Actually <H_C> = <sum (Z_i Z_j - I)/2> for edges. When z_i z_j = -1 (cut edge), term = -1. When same side, term = 0.
# So <H_C> = - <cut_size>. Best cost is minimum <H_C> = -max cut.
# Convert to expected cut value:
best_cut_from_landscape = -best_cost

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(-landscape.T, aspect='auto', origin='lower',
               extent=[gammas[0], gammas[-1], betas[0], betas[-1]],
               cmap='RdPu')
ax.set_xlabel(r'$\gamma$')
ax.set_ylabel(r'$\beta$')
ax.set_title(f'QAOA landscape: expected cut value at p=1\n(max cut on this graph: {max_cut_exact})')
plt.colorbar(im, ax=ax, label='Expected cut')
ax.plot(gamma_opt_lin, beta_opt_lin, 'o', color='cyan', markersize=15,
        markeredgecolor='black', markeredgewidth=2, label=f'Grid best: {best_cut_from_landscape:.2f}')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"Best expected cut from grid scan:  {best_cut_from_landscape:.4f}")
print(f"Exact maximum cut:                 {max_cut_exact}")
print(f"Approximation ratio (p=1 optimum): {best_cut_from_landscape / max_cut_exact:.3f}")

Look at the landscape carefully. Two things stand out:

1. **Multiple local minima.** The cost function is periodic and highly non-convex. Any gradient-based optimizer started at a random point can get stuck.
2. **The optimum's approximation ratio is bounded.** Even at the best $(\gamma, \beta)$, the achieved cut ratio is typically 0.7-0.8. This is the fundamental $p=1$ QAOA limit for random graphs of this density.

To do better, you have to either increase $p$ (deeper circuit) or use a warm start that puts you near a good local minimum.


## Optimize the QAOA parameters

Now use a real optimizer (COBYLA) instead of the grid scan. Track its trajectory across the landscape.


In [ ]:
traj = {'gammas': [], 'betas': [], 'costs': []}

def objective(params):
    g, b = params
    c = qaoa_expected_cost(g, b, N_VERTICES, edges, H_cost)
    traj['gammas'].append(g)
    traj['betas'].append(b)
    traj['costs'].append(c)
    return c


# Try optimization from 5 different random starting points
rng = np.random.default_rng(seed=1)
optima = []
for trial in range(5):
    x0 = np.array([rng.uniform(0, np.pi), rng.uniform(0, np.pi / 2)])
    result = minimize(objective, x0, method='COBYLA', options={'maxiter': 50, 'rhobeg': 0.3})
    optima.append({'trial': trial + 1, 'x0': x0, 'x_opt': result.x, 'cost_opt': result.fun,
                   'cut_opt': -result.fun})

pd.DataFrame(optima)

Each trial converges to a slightly different local minimum, depending on its starting point. This is expected for COBYLA on a non-convex landscape: it finds a local minimum, which may not be the global one.

## Warm start from a classical solution

Instead of starting QAOA from random parameters, we can *warm-start* from a good classical solution. The idea is to use Goemans-Williamson (or any decent classical heuristic) to find a good bit string, then initialize QAOA in a state biased toward that bit string.

There are several ways to encode a classical warm start into QAOA parameters. The simplest, and the most illustrative, is to replace the initial $|+\rangle^{\otimes n}$ with $R_y(2\theta_i)|0\rangle$ on each qubit, where $\theta_i$ depends on the classical solution's bit for qubit $i$.

Specifically: if bit $i$ is 0, set $\theta_i = \epsilon$ (near 0, so the qubit starts near $|0\rangle$). If bit $i$ is 1, set $\theta_i = \pi - \epsilon$ (near $\pi$, so the qubit starts near $|1\rangle$). The small $\epsilon$ keeps the state slightly biased away from a classical basis state, giving QAOA room to improve.


In [ ]:
def warm_start_circuit(classical_bitstring, gamma, beta, N, edges, epsilon=0.25):
    """QAOA circuit with warm start from a classical bit string."""
    qc = QuantumCircuit(N)
    # Warm-start rotations
    for i in range(N):
        theta = np.pi - epsilon if classical_bitstring[i] == '1' else epsilon
        qc.ry(theta, i)
    # QAOA layers (same as before)
    for (i, j) in edges:
        qc.rzz(2 * gamma, i, j)
    for q in range(N):
        qc.rx(2 * beta, q)
    return qc


def warm_start_cost(gamma, beta, classical_bs, N, edges, H_cost):
    qc = warm_start_circuit(classical_bs, gamma, beta, N, edges)
    sv = Statevector.from_instruction(qc)
    return np.real(sv.expectation_value(H_cost))


# Optimize with warm start
if HAVE_GW:
    ws_bs = gw_bitstring
    print(f"Warm-starting from GW solution: {ws_bs} (cut = {cut_size(ws_bs, edges)})")

    def ws_objective(params):
        return warm_start_cost(params[0], params[1], ws_bs, N_VERTICES, edges, H_cost)

    # Multiple starting points
    ws_optima = []
    for trial in range(5):
        x0 = np.array([rng.uniform(0, np.pi), rng.uniform(0, np.pi / 2)])
        result = minimize(ws_objective, x0, method='COBYLA', options={'maxiter': 50, 'rhobeg': 0.3})
        ws_optima.append({'trial': trial + 1, 'x_opt': result.x, 'cost_opt': result.fun,
                          'cut_opt': -result.fun})

    ws_df = pd.DataFrame(ws_optima)
    print("\nWarm-start optima:")
    print(ws_df)

    # Best
    best_ws_trial = ws_df.iloc[ws_df['cut_opt'].idxmax()]
    print(f"\nBest warm-start cut: {best_ws_trial['cut_opt']:.4f} ({best_ws_trial['cut_opt'] / max_cut_exact:.3f} ratio)")
else:
    print("Warm start skipped (no cvxpy). Using cold-start best as reference.")

## Run QAOA on real hardware

Take the best parameters found and measure the QAOA state on real hardware. The measured bit strings will (hopefully) be biased toward good cuts.


In [ ]:
provider = QbraidProvider()
# Devices are named by qBraid QRN. The README lists devices, prices and availability.
DEVICE_ID = 'aws:iqm:qpu:garnet'
device = provider.get_device(DEVICE_ID)

# Use best cold-start parameters
best_trial = min(optima, key=lambda o: o['cost_opt'])
gamma_star, beta_star = best_trial['x_opt']
print(f"Running QAOA on hardware at gamma={gamma_star:.4f}, beta={beta_star:.4f}")

qc_hw = qaoa_circuit(gamma_star, beta_star, N_VERTICES, edges)
qc_hw.measure_all()

# Decompose to a basis the vendor compilers accept. Rigetti's quilc cannot route rzz
# ("Requested to rewire RZZ(...), but we don't know how to do this") and nothing in the
# submission path decomposes it. Verified on Rigetti and IQM; not yet tested on IonQ.
HW_BASIS = ['rz', 'rx', 'ry', 'cz', 'cx', 'h', 'measure']
qc_hw = transpile(qc_hw, basis_gates=HW_BASIS, optimization_level=1)

print(f"Submitting {SHOTS}-shot job to {DEVICE_ID}...")
job = device.run(qc_hw, shots=SHOTS, tags=QUEST_JOB_TAGS)
result = job.result()
counts = result.data.get_counts()

# Compute the expected cut from hardware measurements
total_shots = sum(counts.values())
hw_cut_dist = {}
for bs, count in counts.items():
    # bs is little-endian; reverse for vertex ordering
    bs_vertex = bs[::-1]
    c = cut_size(bs_vertex, edges)
    hw_cut_dist[c] = hw_cut_dist.get(c, 0) + count

hw_expected_cut = sum(c * count for c, count in hw_cut_dist.items()) / total_shots
hw_best_measured = max(hw_cut_dist.keys())

print(f"\nHardware expected cut:  {hw_expected_cut:.3f}")
print(f"Best cut measured:      {hw_best_measured} (achieved by {hw_cut_dist[hw_best_measured]} shots)")
print(f"Ideal expected cut:     {-best_trial['cost_opt']:.3f}")

In [ ]:
# Distribution of measured cuts on hardware
fig, ax = plt.subplots(figsize=(10, 5))
cut_values = sorted(hw_cut_dist.keys())
counts_at_cut = [hw_cut_dist[c] / total_shots for c in cut_values]
bars = ax.bar(cut_values, counts_at_cut, color='#a02580', alpha=0.7, edgecolor='black')

# Highlight the max cut
for c, bar in zip(cut_values, bars):
    if c == max_cut_exact:
        bar.set_color('#2d7a4f')
        bar.set_alpha(1.0)

ax.axvline(hw_expected_cut, color='black', linestyle='--', alpha=0.6,
           label=f'HW expected cut: {hw_expected_cut:.2f}')
ax.axvline(max_cut_exact, color='#2d7a4f', linestyle='-', linewidth=2, alpha=0.8,
           label=f'Exact max cut: {max_cut_exact}')

ax.set_xlabel('Measured cut value')
ax.set_ylabel('Empirical probability')
ax.set_title(f'Distribution of measured cuts on {DEVICE_ID}')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Comparison across methods
Put the numbers on the same table.


In [ ]:
results = [
    {'Method': 'Random cut (expected)', 'Cut value': f'{exp_random:.2f}',
     'Approx ratio': f'{exp_random / max_cut_exact:.3f}'},
    {'Method': 'QAOA p=1, cold start (simulator)', 'Cut value': f'{-best_trial["cost_opt"]:.2f}',
     'Approx ratio': f'{-best_trial["cost_opt"] / max_cut_exact:.3f}'},
]
if HAVE_GW:
    results.append({'Method': 'Goemans-Williamson (classical)', 'Cut value': f'{gw_cut}',
                    'Approx ratio': f'{gw_cut / max_cut_exact:.3f}'})
results.append({'Method': f'QAOA p=1 on hardware ({DEVICE_ID})', 'Cut value': f'{hw_expected_cut:.2f}',
                'Approx ratio': f'{hw_expected_cut / max_cut_exact:.3f}'})
results.append({'Method': 'Exact Max-Cut (enumeration)', 'Cut value': f'{max_cut_exact}',
                'Approx ratio': '1.000'})

pd.DataFrame(results)

On this graph:

- **QAOA at $p=1$ lands in the middle.** Its approximation ratio is above random guessing, which gives 0.64 on this graph, and below Goemans-Williamson.
- **Hardware lowers the result.** The circuit is short, one cost layer and one mixer layer, so noise does not destroy it completely. In our tests IQM Garnet gave an average cut of 10.3 out of a maximum of 14 (a ratio of about 0.74), above the random value of 9.0. On Rigetti Cepheus the result fell below random.
- **Goemans-Williamson is still the strongest classical baseline** for graphs of this size and density, and one-layer QAOA does not beat it. That is the current state of the field.

Doing better than Goemans-Williamson would need either more layers, which means deeper circuits that do not survive on current hardware, or graphs where QAOA has a specific advantage. Finding such graphs is an open research question.

## Where QAOA fits in the bigger picture

QAOA matters academically more than its current numbers suggest. It's a natural NISQ algorithm, shallow and variational and tolerant of noise, and it fits the constraints of today's hardware in a way that many quantum algorithms do not. It also has structural quantum advantages in specific settings. For $p \to \infty$, QAOA can be shown to solve NP-hard problems (if the parameters could be optimized), and for specific problem classes (e.g., MaxCut on certain graph families), constant-depth QAOA has provable performance guarantees that classical algorithms can't match. On top of that, it's a testbed for research on trainability, expressivity, and noise. Many QML lessons transfer from QAOA to other variational algorithms.

The practical question is whether QAOA at experimentally accessible $p$ ever beats the best classical algorithms on any problem instance. As of this notebook's writing, that question is genuinely open. Experimental demonstrations have been done for small instances, but a clear-cut "quantum wins" result is not yet in hand.

For teaching, the pedagogical value is clear. QAOA teaches variational algorithms, noise-resilience trade-offs, and hybrid quantum-classical computation better than most algorithms do. Whether it's the future of optimization is a separate, longer-timeline question.


## Going further
- **Scale up $p$.** Try QAOA at $p=2, 3, 4$ on the same graph. Watch how the ideal approximation ratio improves, and how hardware performance degrades.
- **Different graph structures.** Regular graphs (each vertex has the same degree), sparse graphs, dense graphs, planar graphs. QAOA's performance depends on the graph structure.
- **Compare warm-start strategies.** Use random-cut warm start vs GW warm start vs a greedy warm start. Which converges to better solutions?
- **Apply to other problems.** QAOA works on any Ising-form problem: MaxCut, MaxSAT, Traveling Salesman, portfolio optimization. Formulating your favorite optimization problem as an Ising Hamiltonian is a common student project.
- **Add error mitigation.** Apply zero-noise extrapolation to the hardware measurement. How much of the ideal approximation ratio can you recover?
